# Notebook 04 — Bottleneck & Corridor Audit

Computes betweenness centrality, PageRank, in/out-degree, clustering coefficient.
Identifies Top 5 bottleneck hubs and chronic corridors.
Louvain community detection for zone discovery.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from src.graph.builder import load_graph
from src.graph.analytics import (
    compute_centrality, compute_sla_breach_contribution,
    get_chronic_corridors, detect_communities, bottleneck_report, save_reports
)
print('Analytics modules loaded')

In [ ]:
G = load_graph()
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
centrality_df = compute_centrality(G)
centrality_df = compute_sla_breach_contribution(G, centrality_df)
print('=== TOP 10 BOTTLENECK HUBS ===')
print(centrality_df[['hub','city','betweenness_centrality','pagerank','avg_dwell_min','sla_breach_pct']].head(10).to_string())

In [ ]:
# Betweenness vs SLA breach scatter
fig = px.scatter(
    centrality_df,
    x='betweenness_centrality', y='sla_breach_pct',
    size='trip_volume', color='avg_dwell_min',
    hover_name='hub', color_continuous_scale='RdYlGn_r',
    title='Betweenness Centrality vs SLA Breach % (size = trip volume)',
    template='plotly_dark',
    labels={'betweenness_centrality': 'Betweenness Centrality', 'sla_breach_pct': 'SLA Breach %'}
)
fig.show()
fig.write_html('reports/04_bottleneck_scatter.html')

In [ ]:
chronic_df = get_chronic_corridors(G)
print(f'Total chronic corridors: {len(chronic_df)}')
print('\nTop 10 worst corridors:')
print(chronic_df[['source','destination','route_type','median_delay_ratio','volume']].head(10).to_string())

In [ ]:
# Community detection
partition = detect_communities(G)
if partition:
    community_sizes = pd.Series(partition).value_counts()
    print(f'Communities detected: {len(community_sizes)}')
    print(community_sizes.head(10))

In [ ]:
# Save for dashboard
save_reports(centrality_df, chronic_df)
print('Reports saved to data/processed/')